# BFS & DFS Pattern Playbook - Graphs & Grids
**Topic:** Graphs | **Type:** Pattern notebook (many problems, few templates)

Companion to [`07_graph_traversal.ipynb`](07_graph_traversal.ipynb) (which covers the mechanics: adjacency
lists, plain BFS/DFS, components, cycle detection) and to
[`../01_Generic_Tree/1. bfs_dfs_pattern_playbook.ipynb`](../01_Generic_Tree/1.%20bfs_dfs_pattern_playbook.ipynb)
(the tree-only version of this exact exercise, referenced from section 2.5 of
[`0. tree_traversals_dfs_and_bfs.ipynb`](../01_Generic_Tree/0.%20tree_traversals_dfs_and_bfs.ipynb)).

**The idea, unchanged from the tree notebook:** once you own **one baseline template per pattern**, a large
slice of Blind 75 / NeetCode 150 stops being "N different algorithms to memorise" and becomes "1 template,
plugged in N different ways." The tree notebook proved this on `TreeNode`s. This notebook proves the exact
same templates carry over unchanged to **grids** (implicit graphs) and **explicit graphs** (adjacency lists) -
the only thing that ever changes is *what counts as a node* and *what counts as a neighbour*.

**How to read this notebook:** every variant cell starts with **"Same as baseline, except:"** - read that
line first. If you can already predict the diff before reading the code, you've internalised the pattern.

- **Part A - BFS:** one generic function, `bfs_generic`, solves **7** different problems by varying its
  3 arguments - grids, an implicit word-graph, a board-as-graph, and an explicit adjacency-list graph.
- **Part B - DFS:** two templates - **backtracking** (**4** problems) and **visited-marking graph
  traversal** (**4** problems).


---
# Part A - BFS: One Generic Template Solves Them All

**The insight:** every BFS problem is really the same question - *"starting from some set of nodes, spread
outward one edge at a time, and either (a) stop the instant you meet a target, or (b) record how far every
node is from its nearest starting point."* That's it. Grids, word ladders, board games, and explicit graphs
are all just different answers to "what counts as a node, and what counts as a neighbour" - the exact same
question the tree notebook answered with "a tree node, and its children."

So instead of writing BFS seven separate times, we write it **once**, parameterised by three things:

| Parameter | Answers |
|---|---|
| `sources` | Where does the ripple start? (one node, or many at once - "multi-source BFS") |
| `neighbors_fn(node)` | What counts as one step away from a node? |
| `is_target_fn(node)` (optional) | Should we stop early the instant we find a match? |


In [ ]:
from collections import deque

def bfs_generic(sources, neighbors_fn, is_target_fn=None):
    """One BFS to rule them all.

    - sources: iterable of starting nodes (single-source BFS = [start]).
    - neighbors_fn(node) -> iterable of nodes one step away from `node`.
    - is_target_fn(node) -> bool, optional early-stop condition.

    Returns:
      - if is_target_fn is given: the number of EDGES from the nearest source to the
        first node that satisfies it, or -1 if no reachable node ever does.
      - otherwise: a dict {node: distance_from_nearest_source} for every reached node.
    """
    dist = {s: 0 for s in sources}         # every source starts at distance 0 - this IS multi-source BFS
    q = deque(sources)
    while q:
        node = q.popleft()
        if is_target_fn is not None and is_target_fn(node):
            return dist[node]              # early stop - guaranteed to be the NEAREST match
        for nxt in neighbors_fn(node):
            if nxt not in dist:            # dist doubles as the visited set
                dist[nxt] = dist[node] + 1
                q.append(nxt)
    return -1 if is_target_fn is not None else dist


# Toy sanity check: a tiny 4-node chain graph, single source.
toy_graph = {"A": ["B"], "B": ["A", "C"], "C": ["B", "D"], "D": ["C"]}
print(bfs_generic(["A"], lambda n: toy_graph[n]))                       # distances from A to everyone
print(bfs_generic(["A"], lambda n: toy_graph[n], lambda n: n == "D"))   # steps from A to D


## A.1 - LC 542: 01 Matrix

**Same as baseline, except:** `sources` becomes **every** `0` cell at once (multi-source - the ripple
starts simultaneously from all of them, which is exactly what "distance to the *nearest* zero" needs).
`neighbors_fn` is grid movement in 4 directions. No `is_target_fn` - we want the *full* distance map, not
an early stop. The grid is an **implicit graph**: cells are nodes, orthogonal adjacency is the edge set -
nothing is ever built into an actual adjacency list.


In [ ]:
def update_matrix(mat):
    """For every cell, the distance to its nearest 0 cell."""
    rows, cols = len(mat), len(mat[0])
    sources = [(r, c) for r in range(rows) for c in range(cols) if mat[r][c] == 0]   # ALL zeros at once

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                yield (nr, nc)

    dist = bfs_generic(sources, neighbors)             # no is_target_fn -> full distance dict
    return [[dist[(r, c)] for c in range(cols)] for r in range(rows)]


mat = [[0, 0, 0], [0, 1, 0], [1, 1, 1]]
print(update_matrix(mat))                  # expect [[0,0,0],[0,1,0],[1,2,1]]


## A.2 - LC 286: Walls and Gates

**Same as baseline, except:** `sources` = every gate (`0` cell), and `neighbors_fn` now **blocks movement
through walls** (`-1` cells) - the only real difference from 01 Matrix. We then write the resulting
distances straight back into the grid (in place, as the problem asks), leaving any cell BFS never reached
(a room sealed off by walls) at its original `INF` sentinel.


In [ ]:
INF = 2**31 - 1

def walls_and_gates(rooms):
    rows, cols = len(rooms), len(rooms[0])
    sources = [(r, c) for r in range(rows) for c in range(cols) if rooms[r][c] == 0]   # ALL gates at once

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and rooms[nr][nc] != -1:   # walls block movement
                yield (nr, nc)

    dist = bfs_generic(sources, neighbors)
    for (r, c), d in dist.items():
        rooms[r][c] = d                        # write distances back in place
    return rooms


rooms = [[INF, -1, 0, INF],
         [INF, INF, INF, -1],
         [INF, -1, INF, -1],
         [0, -1, INF, INF]]
for row in walls_and_gates(rooms):
    print(row)


## A.3 - LC 994: Rotting Oranges

**Same as baseline, except:** `sources` = every already-rotten orange (multi-source: they all spread at
the same time, and each BFS "layer" is exactly one unit of elapsed time). `neighbors_fn` only steps onto
*fresh* cells. The answer is then read out of the distance dict rather than returned directly: if any fresh
orange was never reached, return `-1`; otherwise the answer is the **maximum** distance recorded (the last
orange to rot).


In [ ]:
def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    sources = [(r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 2]   # ALL rotten at once
    fresh_total = sum(1 for r in range(rows) for c in range(cols) if grid[r][c] == 1)
    if fresh_total == 0:
        return 0

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:   # step only onto FRESH cells
                yield (nr, nc)

    dist = bfs_generic(sources, neighbors)
    rotted_fresh = [d for (r, c), d in dist.items() if grid[r][c] == 1]
    if len(rotted_fresh) < fresh_total:
        return -1                              # some fresh orange was unreachable
    return max(rotted_fresh)                   # time = the LAST orange to rot


print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))   # expect 4
print(oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]))   # expect -1 (unreachable fresh orange)


## A.4 - LC 127: Word Ladder

**Same as baseline, except:** the "graph" is now fully **implicit** - nodes are words, and a neighbour is
any word in the dictionary that differs by exactly one letter (generated on the fly by trying all 26
letters at every position - no adjacency list is ever materialised). `is_target_fn` checks "did we reach
`endWord`?" for the early stop. The answer LeetCode wants is the number of *words* in the ladder, which is
`edges + 1`.


In [ ]:
def word_ladder(begin_word, end_word, word_list):
    words = set(word_list)
    if end_word not in words:
        return 0

    def neighbors(word):                       # one step away = one letter changed, still a real word
        for i in range(len(word)):
            for ch in "abcdefghijklmnopqrstuvwxyz":
                if ch != word[i]:
                    cand = word[:i] + ch + word[i+1:]
                    if cand in words:
                        yield cand

    steps = bfs_generic([begin_word], neighbors, lambda w: w == end_word)
    return steps + 1 if steps != -1 else 0


print(word_ladder("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]))   # expect 5
print(word_ladder("hit", "cog", ["hot", "dot", "dog", "lot", "log"]))          # expect 0 (cog missing)


## A.5 - LC 909: Snakes and Ladders

**Same as baseline, except:** the "graph" is a board. A node is a square number (1..n^2); a neighbour is
any square reachable by rolling a die 1-6 **and then resolving a snake/ladder jump if the landing square has
one**. `is_target_fn` checks "did we reach the last square?" The board is stored row-major, but squares are
numbered in a **boustrophedon** (back-and-forth) pattern, so `cell_value` does the number-to-(row,col)
conversion once, and everything else is the same template.


In [ ]:
def snakes_and_ladders(board):
    n = len(board)

    def cell_value(square):
        """1-indexed square number -> the board's value at that square (boustrophedon layout)."""
        square -= 1
        r, c = divmod(square, n)
        if r % 2 == 1:              # every other row runs right-to-left
            c = n - 1 - c
        row = n - 1 - r             # square 1 is the BOTTOM row
        return board[row][c]

    def neighbors(square):
        for d in range(1, 7):       # a die roll of 1..6
            nxt = square + d
            if nxt > n * n:
                continue
            v = cell_value(nxt)
            yield v if v != -1 else nxt   # -1 means "no snake/ladder here" -> stay on nxt

    return bfs_generic([1], neighbors, lambda s: s == n * n)


board = [[-1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1],
         [-1, 35, -1, -1, 13, -1],
         [-1, -1, -1, -1, -1, -1],
         [-1, 15, -1, -1, -1, -1]]
print(snakes_and_ladders(board))          # expect 4


## A.6 - LC 200: Number of Islands (BFS flavour)

**Same as baseline, except:** this isn't a shortest-path question at all - it's "how many separate blobs of
land are there?" But the *mechanism* for flood-filling one blob is still `bfs_generic`: run it from a single
unvisited land cell with no `is_target_fn`, and the returned dict's keys are exactly "every cell in this
island." Repeat once per undiscovered land cell and count how many times you had to start over.

This is the section 2.5 decision-table rule made concrete: no shortest-path requirement, so either BFS or
DFS works - BFS is shown here because it reuses `bfs_generic` directly; the DFS version appears in Part B
for a direct side-by-side comparison.


In [ ]:
def num_islands_bfs(grid):
    rows, cols = len(grid), len(grid[0])
    visited = set()

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                yield (nr, nc)

    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in visited:
                count += 1
                reached = bfs_generic([(r, c)], neighbors)   # flood-fills exactly one island
                visited |= reached.keys()
    return count


grid1 = [list("11110"), list("11010"), list("11000"), list("00000")]
print(num_islands_bfs(grid1))             # expect 1

grid2 = [list("11000"), list("11000"), list("00100"), list("00011")]
print(num_islands_bfs(grid2))             # expect 3


## A.7 - LC 133: Clone Graph (BFS flavour)

**Same skeleton, one extra job:** this is the first problem in this notebook on an **explicit** graph
(nodes carry a `.neighbors` list directly, the way `07_graph_traversal.ipynb` builds an adjacency list).
The queue-plus-visited-dict shape is identical to every grid problem above, but here `visited` (renamed
`clones`) doesn't just store "have I seen this node" - it stores **the cloned copy** of each original node.
Every time we discover a new neighbour we (a) create its clone and (b) wire it onto the current node's
clone. It's `bfs_generic` with the loop body doing a bit more bookkeeping.


In [ ]:
class GNode:
    def __init__(self, val, neighbors=None):
        self.val = val
        self.neighbors = neighbors or []


def clone_graph_bfs(node):
    if not node:
        return None
    clones = {node: GNode(node.val)}           # visited dict IS the clone map
    q = deque([node])
    while q:
        cur = q.popleft()
        for nei in cur.neighbors:
            if nei not in clones:
                clones[nei] = GNode(nei.val)
                q.append(nei)
            clones[cur].neighbors.append(clones[nei])   # wire the clone edges as we go
    return clones[node]


a, b, c = GNode(1), GNode(2), GNode(3)
a.neighbors = [b, c]; b.neighbors = [a, c]; c.neighbors = [a, b]
cloned = clone_graph_bfs(a)
print("cloned root value:", cloned.val)
print("cloned neighbours:", sorted(n.val for n in cloned.neighbors))
print("is a real copy (not the same object):", cloned is not a)


## A.8 - Recap: what actually changed each time

| Problem | `sources` | `neighbors_fn` | `is_target_fn` | Answer shape |
|---|---|---|---|---|
| 542 01 Matrix | every `0` cell | 4-directional | *(none)* | full distance grid |
| 286 Walls & Gates | every gate | 4-directional, blocked by walls | *(none)* | distances written in place |
| 994 Rotting Oranges | every rotten cell | 4-directional, only onto fresh | *(none)* | `max(distances)` or `-1` |
| 127 Word Ladder | `[beginWord]` | 1-letter-different words in dict | `== endWord` | `edges + 1` |
| 909 Snakes & Ladders | `[1]` | dice roll + snake/ladder resolve | `== n*n` | `edges` |
| 200 Num Islands | one unvisited land cell (repeated) | 4-directional land cells | *(none)* | count of calls |
| 133 Clone Graph | `[node]` | explicit `.neighbors` list | *(none)* | clone map built during the walk |

Every row is the **same nine lines of `bfs_generic`** underneath, whether the "graph" is a grid, a
dictionary of words, a board, or an actual adjacency-list structure. What you're learning per problem is
"what is a node here, and what is a neighbour" - not a new algorithm.


---
# Part B - DFS: Two Templates for Two Kinds of Questions

Just like on trees, DFS here doesn't collapse into one parameterised function - the *shape* of the
recursion depends on what the question is asking:

| Template | Shape | Answers questions like |
|---|---|---|
| **B.1 Backtracking** | `choose -> recurse -> un-choose` in a loop over candidates | subsets, permutations, combinations, word search |
| **B.2 Visited-marking** | `if invalid or seen: return`; `mark`; recurse into neighbours | connected components, cycle detection, multi-region flood fill |

(The tree notebook's **third** DFS template - "return a value up from children" - doesn't reappear here:
it's a tree-shaped question, since it relies on a node having a small, fixed number of children to combine
answers from. On a general graph "combine the answers of my neighbours" runs straight into cycles, which is
exactly why graph DFS problems are about *marking* reachability/state instead.)


## B.1 - Backtracking DFS

**Baseline - LC 78: Subsets.** The shape: **choose** one candidate, **recurse** with it included, then
**un-choose** it (undo) before trying the next candidate. `path` is mutated in place and copied only when
recorded - the same "call stack IS the state machine" idea DFS always relies on, just with an explicit
`path` list riding along.


In [ ]:
def subsets(nums):
    out, path = [], []
    def backtrack(start):
        out.append(path[:])                    # every prefix along the way is a valid subset
        for i in range(start, len(nums)):
            path.append(nums[i])                # CHOOSE
            backtrack(i + 1)                     # RECURSE (only look forward - avoids duplicates)
            path.pop()                           # UN-CHOOSE
    backtrack(0)
    return out


print(subsets([1, 2, 3]))


### B.1a - LC 46: Permutations

**Same as baseline, except:** order matters now, so we can't just "only look forward" (`start` index) -
every unused element is a valid next choice, from anywhere in the list. That needs a `used[]` array instead
of a start index, and the base case is "path is full length" instead of "record every prefix."


In [ ]:
def permutations(nums):
    out, path = [], []
    used = [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            out.append(path[:]); return         # base case: only full-length paths count
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True                       # CHOOSE
            path.append(nums[i])
            backtrack()                           # RECURSE
            path.pop()                            # UN-CHOOSE
            used[i] = False
    backtrack()
    return out


print(permutations([1, 2, 3]))


### B.1b - LC 39: Combination Sum

**Same as baseline, except:** candidates can be **reused**, so the recursive call passes `i` (not `i + 1`)
- staying at the same index keeps that candidate eligible again. We also prune: stop the instant
`remaining < 0`, and record only when `remaining == 0` exactly.


In [ ]:
def combination_sum(candidates, target):
    out, path = [], []
    def backtrack(start, remaining):
        if remaining == 0:
            out.append(path[:]); return
        if remaining < 0:
            return                               # prune - this branch overshot
        for i in range(start, len(candidates)):
            path.append(candidates[i])            # CHOOSE
            backtrack(i, remaining - candidates[i])   # RECURSE (i, not i+1 - reuse allowed)
            path.pop()                             # UN-CHOOSE
    backtrack(0, target)
    return out


print(combination_sum([2, 3, 6, 7], 7))    # expect [[2, 2, 3], [7]]


### B.1c - LC 79: Word Search

**Same as baseline, except:** the "candidates" are grid directions instead of a fixed list, and the
un-choose step restores a **grid cell** (temporarily overwritten with a marker so we don't revisit it in
this same path) instead of popping a list. Same choose/recurse/un-choose rhythm - it just runs over
`(row, col)` moves instead of array indices.


In [ ]:
def word_search(board, word):
    rows, cols = len(board), len(board[0])
    def backtrack(r, c, i):
        if i == len(word):
            return True                          # matched every letter
        if r < 0 or r >= rows or c < 0 or c >= cols or board[r][c] != word[i]:
            return False
        tmp = board[r][c]
        board[r][c] = "#"                        # CHOOSE: mark this cell as "in use" for this path
        found = (backtrack(r + 1, c, i + 1) or backtrack(r - 1, c, i + 1) or
                 backtrack(r, c + 1, i + 1) or backtrack(r, c - 1, i + 1))   # RECURSE
        board[r][c] = tmp                        # UN-CHOOSE: restore the cell for other paths
        return found
    for r in range(rows):
        for c in range(cols):
            if backtrack(r, c, 0):
                return True
    return False


board = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
print(word_search([row[:] for row in board], "ABCCED"))   # expect True
print(word_search([row[:] for row in board], "SEE"))      # expect True
print(word_search([row[:] for row in board], "ABCB"))     # expect False


## B.2 - Visited-marking Graph DFS

**Baseline - LC 200: Number of Islands (DFS flavour).** The shape: `if out of bounds / wrong value /
already seen: return`; otherwise `mark` this cell and recurse into every neighbour. Compare this to
`num_islands_bfs` in Part A - it is the **exact same algorithm**, just walked with the call stack instead
of an explicit queue. Neither container changes the answer, because this problem never asked for a
*shortest* anything.


In [ ]:
def num_islands_dfs(grid):
    rows, cols = len(grid), len(grid[0])
    visited = set()
    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != "1" or (r, c) in visited:
            return
        visited.add((r, c))                      # MARK
        dfs(r + 1, c); dfs(r - 1, c); dfs(r, c + 1); dfs(r, c - 1)   # recurse into every neighbour
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in visited:
                count += 1
                dfs(r, c)
    return count


grid1 = [list("11110"), list("11010"), list("11000"), list("00000")]
print(num_islands_dfs(grid1))          # expect 1

grid2 = [list("11000"), list("11000"), list("00100"), list("00011")]
print(num_islands_dfs(grid2))          # expect 3


### B.2a - LC 207: Course Schedule

**Same as baseline, except:** "visited" isn't a plain set anymore - it's a **3-state colouring**
(`WHITE` = untouched, `GRAY` = currently on the call stack, `BLACK` = fully finished). A cycle is detected
the instant DFS walks back into a `GRAY` node - that means it's still one of *its own* ancestors on the
current path, i.e. a back edge. This 3-colour trick is the standard way DFS answers "is there a cycle" on a
directed graph, and it directly reuses the "if invalid/seen: return; mark; recurse" skeleton. It's also the
first time this notebook needed a **directed** graph - `07_graph_traversal.ipynb`'s cycle check
(`has_cycle_undirected`) only tracks a single `parent`, because an *undirected* edge back to your own parent
is not a cycle; on a directed graph you instead need the full 3-colour state to tell "still open" from
"already closed."


In [ ]:
def can_finish(num_courses, prerequisites):
    graph = {i: [] for i in range(num_courses)}
    for course, pre in prerequisites:
        graph[pre].append(course)
    WHITE, GRAY, BLACK = 0, 1, 2
    state = [WHITE] * num_courses
    def dfs(node):
        if state[node] == GRAY:
            return False                          # back edge to a node still on the stack -> CYCLE
        if state[node] == BLACK:
            return True                            # already fully explored and safe - skip re-checking
        state[node] = GRAY                         # MARK as "in progress"
        for nxt in graph[node]:
            if not dfs(nxt):
                return False
        state[node] = BLACK                        # MARK as "fully safe"
        return True
    return all(dfs(i) for i in range(num_courses))


print(can_finish(2, [[1, 0]]))              # expect True  (0 -> 1, no cycle)
print(can_finish(2, [[1, 0], [0, 1]]))      # expect False (0 -> 1 -> 0, a cycle)


### B.2b - LC 417: Pacific Atlantic Water Flow

**Same as baseline, except:** we run the *same* visited-marking DFS **twice** - once flooding inward from
every Pacific-adjacent border cell, once from every Atlantic-adjacent border cell - with the direction of
"valid neighbour" flipped (water flows from high ground to low or equal ground, so flooding *backward* from
the ocean means only climbing to **equal-or-higher** cells is allowed). A cell that shows up in **both**
visited sets can reach both oceans.


In [ ]:
def pacific_atlantic(heights):
    if not heights:
        return []
    rows, cols = len(heights), len(heights[0])
    pacific, atlantic = set(), set()
    def dfs(r, c, visited, prev_height):
        if r < 0 or r >= rows or c < 0 or c >= cols or (r, c) in visited or heights[r][c] < prev_height:
            return                                # can't flow "uphill" from the ocean's perspective
        visited.add((r, c))                       # MARK
        dfs(r + 1, c, visited, heights[r][c]); dfs(r - 1, c, visited, heights[r][c])
        dfs(r, c + 1, visited, heights[r][c]); dfs(r, c - 1, visited, heights[r][c])
    for c in range(cols):
        dfs(0, c, pacific, heights[0][c])          # top row touches the Pacific
        dfs(rows - 1, c, atlantic, heights[rows - 1][c])   # bottom row touches the Atlantic
    for r in range(rows):
        dfs(r, 0, pacific, heights[r][0])          # left column touches the Pacific
        dfs(r, cols - 1, atlantic, heights[r][cols - 1])   # right column touches the Atlantic
    return [list(cell) for cell in pacific & atlantic]


heights = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]
print(sorted(pacific_atlantic(heights)))


### B.2c - LC 133: Clone Graph (DFS flavour)

**Same as baseline, except:** exactly like Part A.7's BFS version, the visited structure (`clones`) doubles
as the clone map - only now the traversal is recursive instead of queue-driven. Comparing this to
`clone_graph_bfs` side by side is the clearest possible demonstration of the section 2.5 rule: when there's
no shortest-path requirement, BFS and DFS are interchangeable, and the choice comes down to which container
you'd rather write.


In [ ]:
def clone_graph_dfs(node, clones=None):
    if not node:
        return None
    if clones is None:
        clones = {}
    if node in clones:
        return clones[node]                        # already visited -> return the existing clone
    clones[node] = GNode(node.val)                  # MARK (create the clone BEFORE recursing - breaks cycles)
    for nei in node.neighbors:
        clones[node].neighbors.append(clone_graph_dfs(nei, clones))
    return clones[node]


a2, b2, c2 = GNode(1), GNode(2), GNode(3)
a2.neighbors = [b2, c2]; b2.neighbors = [a2, c2]; c2.neighbors = [a2, b2]
cloned2 = clone_graph_dfs(a2)
print("cloned root value:", cloned2.val)
print("cloned neighbours:", sorted(n.val for n in cloned2.neighbors))
print("is a real copy (not the same object):", cloned2 is not a2)


## B.3 - Recap: which template, and what changed

| Problem | Template | Same as baseline, except... |
|---|---|---|
| 78 Subsets | B.1 backtracking | *(this IS the baseline)* |
| 46 Permutations | B.1 backtracking | swap the `start` index for a `used[]` array - order matters now |
| 39 Combination Sum | B.1 backtracking | recurse with `i` not `i+1` (reuse allowed) + prune on `remaining < 0` |
| 79 Word Search | B.1 backtracking | candidates are grid moves; un-choose restores a grid cell, not a list |
| 200 Num Islands (DFS) | B.2 visited-marking | *(this IS the baseline)* |
| 207 Course Schedule | B.2 visited-marking | 3-state colouring instead of a plain visited set, to catch cycles |
| 417 Pacific Atlantic | B.2 visited-marking | run it twice, with the "valid neighbour" direction flipped |
| 133 Clone Graph (DFS) | B.2 visited-marking | the visited set doubles as the clone map |


## 🧩 Patterns Learned

- **BFS collapses into one function, everywhere.** `bfs_generic(sources, neighbors_fn, is_target_fn)`
  solved every tree BFS problem in the previous notebook, and now 7 more across grids, an implicit
  word-graph, a board, and an explicit adjacency-list graph. The skill isn't "know 7 algorithms" - it's
  "identify what a node and a neighbour are, and whether you need an early stop."
- **Multi-source BFS is not a special algorithm - it's just `sources` having more than one element.**
  01 Matrix, Walls and Gates, and Rotting Oranges are the same function with different starting sets. This
  is the one BFS trick that essentially never shows up on trees (a tree only has one natural root to start
  from) but is extremely common on grids.
- **Graph DFS drops the tree notebook's "return-a-value" template** and keeps two others:
  - **backtracking** (`choose -> recurse -> un-choose`) for anything that enumerates combinations/paths -
    identical in shape to the tree notebook's path-backtracking template, just choosing from array indices
    or grid moves instead of tree children;
  - **visited-marking** (`if invalid/seen: return; mark; recurse`) for anything about reachability,
    components, or cycles - this one is graph-specific, because it's what a visited set has to do once
    edges can point back on themselves.
- **BFS and DFS are interchangeable exactly when there's no shortest-path requirement.** Number of Islands
  and Clone Graph were solved both ways in this notebook with identical results - the only real question is
  which container (queue vs. call stack) you'd rather write.
- **Undirected cycle detection (a `parent` check, in `07_graph_traversal.ipynb`) is not the same trick as
  directed cycle detection (3-colour DFS, section B.2a).** An undirected edge back to your parent is normal,
  not a cycle - a directed edge back to any node still "in progress" always is.
- **This is the same lesson as the tree notebook's sections 2.2->2.3 and 2.5, one level up, on a new data
  shape:** own the *template*, and the *variant* is a one-or-two-line diff you can predict before you write
  it - regardless of whether the underlying structure is a tree, a grid, or an explicit graph.
